# Example Notebook for WIMP Analysis

## Setup

In [2]:
from pathlib import Path
import json, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## Halo

In [4]:
from wimp_legend.halo import StandardHaloModel, velocity_moment
from wimp_legend.kinematics import minimum_speed_c

m_chi_GeV = 30
m_target_GeV = 70

halo = StandardHaloModel.from_km_s()
E = np.geomspace(0.1, 50, 50) # keV_nr
vmin = minimum_speed_c(E, m_chi_GeV, m_target_GeV) # beta = v/c
eta = velocity_moment(
halo.lab_speed_pdf, halo.max_lab_speed, vmin, n=-1,
breakpoints=halo.integration_breakpoints)


## WIMP Model

In [5]:
from wimp_legend.interactions import (
WilsonCoefficients, particle_response, RESPONSE_CHANNELS)
coeff = WilsonCoefficients.from_proton_neutron({
1: (1e-6, 1e-6), # O1: equal proton/neutron coupling
4: (2e-6, 0.0), # O4: proton-only coupling
})
assert coeff.get(1, 0) == 1e-6
assert coeff.get(1, 1) == 0.0
R = particle_response('M', coeff, tau=0, tau_prime=0,
Q=1e-3, V=1e-6, j_chi=0.5)

In [ ]:
coeff

WilsonCoefficients(values=mappingproxy({1: (1e-06, 0.0), 3: (0.0, 0.0), 4: (1e-06, 1e-06), 5: (0.0, 0.0), 6: (0.0, 0.0), 7: (0.0, 0.0), 8: (0.0, 0.0), 9: (0.0, 0.0), 10: (0.0, 0.0), 11: (0.0, 0.0), 12: (0.0, 0.0), 13: (0.0, 0.0), 14: (0.0, 0.0), 15: (0.0, 0.0)}))

## Nuclear Responses

In [7]:
from wimp_legend.targets import (
SourceReference, Isotope,
germanium_afh2014_momentum, load_germanium_afh2014)

# Demonstration only: correct rest energy needed
synthetic = SourceReference(
reference='illustrative Ge-76 specification',
locator='example only', kind='synthetic')

ge76 = Isotope(symbol='Ge', A=76, Z=32,
mass_GeV=76 * 0.93149410242, spin=0.0,
sources={x: synthetic for x in ('identity','mass_GeV','spin')})

y = germanium_afh2014_momentum(ge76.A)
dataset = load_germanium_afh2014(ge76, momentum=y)
W00 = dataset.evaluate(
channel='M', tau=0, tau_prime=0, q2_GeV2=0.001)

## Rate Assembly

In [8]:
from wimp_legend.rates import (
NREFTRateConfig, differential_rate_per_kg_day_keV)
from wimp_legend.interactions import WilsonCoefficients

coeff = WilsonCoefficients.from_proton_neutron({1: (1e-6, 1e-6)})
config = NREFTRateConfig(m_N_reference_GeV=0.939) # declared

rate = differential_rate_per_kg_day_keV(
[0.5, 1.0, 2.0], m_chi_GeV=30.0,
rho_chi_GeV_cm3=0.3, targets_per_kg=6.0e24,
dataset=dataset, coefficients=coeff, j_chi=0.5,
config=config, speed_pdf=halo.lab_speed_pdf,
beta_max=halo.max_lab_speed,
breakpoints=halo.integration_breakpoints)


## Detector Response

In [ ]:
from wimp_legend.detector import (
DetectorResponse, ElectronRecoilResponse,
LindhardQuenching, GaussianResolution, ConstantEfficiency)

nr_response = DetectorResponse(
quenching=LindhardQuenching(Z=32, k=0.157),
resolution=GaussianResolution(0.1),
observed_efficiency=ConstantEfficiency(1))

er_response = ElectronRecoilResponse(
resolution=GaussianResolution(0.1),
observed_efficiency=ConstantEfficiency(1))

edges = [0.1, 0.3, 0.6, 1.0, 2.0]
S = nr_response.expected_counts(
lambda E: 1.0, edges, 100.0, (0.0, 20.0))
p = nr_response.bin_probabilities(5.0, edges)

assert sum(p) <= 1.0 + 1e-12 # events outside window are lost
